# ✈️ Avionics Certification Assistant
### DO-178C | DO-254 | ARP4754A | ARP4761

This notebook implements a certification-aware assistant using:
- Groq (llama-3.1-8b-instant)
- LangGraph (LangRAG flow)
- Serper API (retrieval)
- Gradio UI

In [ ]:
!pip install langchain langgraph langchain-community langchain-core gradio groq google-search-results

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
import gradio as gr

## 🔑 Set API Keys

In [ ]:
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"
os.environ["SERPER_API_KEY"] = "YOUR_SERPER_API_KEY"

## 🤖 LLM Setup (Groq)

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

## 🔍 Retrieval (Serper)

In [ ]:
search = SerpAPIWrapper()

## 🔄 LangGraph Workflow

In [ ]:
class GraphState(dict):
    pass

def retrieve(state):
    query = state["question"]
    docs = search.run(query)
    return {"context": docs, "question": query}

def generate(state):
    context = state["context"]
    question = state["question"]

    prompt = f"""
You are an avionics certification expert.

Follow strict format:

ANSWER:
<clear explanation>

STANDARD REFERENCE:
<standards>

EXAMPLE:
<if applicable>

NOTES:
<audit insights>

Question:
{question}

Context:
{context}
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"response": response.content}

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app_graph = workflow.compile()

## 🎛️ Gradio UI

In [ ]:
def avionics_assistant(user_input):
    result = app_graph.invoke({"question": user_input})
    return result["response"]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # ✈️ Avionics Certification Assistant
    ### DO-178C | DO-254 | ARP4754A | ARP4761
    """)

    user_input = gr.Textbox(label="Ask a certification question")
    output = gr.Textbox(label="Response", lines=20)

    btn = gr.Button("Analyze")
    btn.click(avionics_assistant, inputs=user_input, outputs=output)

demo.launch()